In [13]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]

for t in thresholds:
    predicted = (transcript_df["similarity"] >= t).astype(int)
    accuracy = (predicted == transcript_df["human_label"]).mean()
    precision = precision_score(transcript_df["human_label"], predicted, zero_division=0)
    recall = recall_score(transcript_df["human_label"], predicted, zero_division=0)
    print(f"Threshold {t}: accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

Threshold 0.6: accuracy=0.960  precision=0.977  recall=0.977
Threshold 0.7: accuracy=0.920  precision=0.976  recall=0.930
Threshold 0.75: accuracy=0.900  precision=0.975  recall=0.907
Threshold 0.8: accuracy=0.880  precision=0.974  recall=0.884
Threshold 0.85: accuracy=0.880  precision=1.000  recall=0.860
Threshold 0.9: accuracy=0.840  precision=1.000  recall=0.814


In [15]:
import pandas as pd

transcript_df = pd.read_csv("../data/pipeline_results.csv")
print(transcript_df.shape)
print(transcript_df.columns.tolist())


(50, 9)
['passage', 'question', 'answer', 'answer_word_count', 'audio_path', 'transcript', 'similarity', 'human_label', 'prediction']


In [17]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]

for t in thresholds:
    predicted = (transcript_df["similarity"] >= t).astype(int)
    accuracy = (predicted == transcript_df["human_label"]).mean()
    precision = precision_score(transcript_df["human_label"], predicted, zero_division=0)
    recall = recall_score(transcript_df["human_label"], predicted, zero_division=0)
    print(f"Threshold {t}: accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

Threshold 0.6: accuracy=0.960  precision=0.977  recall=0.977
Threshold 0.7: accuracy=0.920  precision=0.976  recall=0.930
Threshold 0.75: accuracy=0.900  precision=0.975  recall=0.907
Threshold 0.8: accuracy=0.880  precision=0.974  recall=0.884
Threshold 0.85: accuracy=0.880  precision=1.000  recall=0.860
Threshold 0.9: accuracy=0.840  precision=1.000  recall=0.814


In [19]:
def classify(similarity):
    if similarity >= 0.65:
        return "Correct"
    elif similarity <= 0.55:
        return "Incorrect"
    else:
        return "Borderline"

transcript_df["prediction"] = transcript_df["similarity"].apply(classify)
transcript_df.to_csv("../data/pipeline_results.csv", index=False)

print(transcript_df["prediction"].value_counts())

prediction
Correct       41
Incorrect      7
Borderline     2
Name: count, dtype: int64


In [21]:
borderline = transcript_df[transcript_df["prediction"] == "Borderline"]
print(borderline[["answer", "transcript", "similarity", "human_label"]])

               answer          transcript  similarity  human_label
13      Pleurobrachia         Plorobracia    0.639905            1
38  William Stranahan  William Strenohin.    0.647472            1


In [30]:
import pandas as pd

df = pd.read_csv("../data/pipeline_results.csv")
print(df.shape)

errors = df[df["prediction"].isin(["Incorrect", "Borderline"])]
print(errors[["answer", "transcript", "similarity", "prediction"]])

(50, 9)
                     answer                        transcript  similarity  \
2            Lower Lorraine                   Lower the rain.    0.279233   
4                 10,000 m2             10,000 square meters.    0.000000   
10                  Beyoncé                        Be on say.    0.167976   
13            Pleurobrachia                       Plorobracia    0.639905   
27  Mi'kmaq and the Abenaki  Me, Kamek, and I'll be knocking.    0.222933   
32             gas turbines                    Gast her buns.    0.159867   
35                  Thoreau                             Dora.    0.261759   
38        William Stranahan                William Strenohin.    0.647472   
40                 Nintendo                               念獵獵    0.131477   

    prediction  
2    Incorrect  
4    Incorrect  
10   Incorrect  
13  Borderline  
27   Incorrect  
32   Incorrect  
35   Incorrect  
38  Borderline  
40   Incorrect  


In [32]:

errors = df[df["prediction"].isin(["Incorrect", "Borderline"])]
print(errors[["answer", "transcript", "similarity", "prediction"]])

                     answer                        transcript  similarity  \
2            Lower Lorraine                   Lower the rain.    0.279233   
4                 10,000 m2             10,000 square meters.    0.000000   
10                  Beyoncé                        Be on say.    0.167976   
13            Pleurobrachia                       Plorobracia    0.639905   
27  Mi'kmaq and the Abenaki  Me, Kamek, and I'll be knocking.    0.222933   
32             gas turbines                    Gast her buns.    0.159867   
35                  Thoreau                             Dora.    0.261759   
38        William Stranahan                William Strenohin.    0.647472   
40                 Nintendo                               念獵獵    0.131477   

    prediction  
2    Incorrect  
4    Incorrect  
10   Incorrect  
13  Borderline  
27   Incorrect  
32   Incorrect  
35   Incorrect  
38  Borderline  
40   Incorrect  


In [34]:
manifest = pd.read_csv("../data/audio_manifest.csv")
nintendo_row = manifest[manifest["answer"] == "Nintendo"]
print(nintendo_row[["answer", "audio_path"]])

      answer                         audio_path
40  Nintendo  data/audio/pyttsx3/pyttsx3_40.mp3


In [36]:
merged = df.merge(manifest[["answer", "audio_path"]], on="answer", how="left")
merged["tts_engine"] = merged["audio_path"].apply(lambda p: "gtts" if "gtts" in p else "pyttsx3")

print(merged.groupby("tts_engine")["prediction"].value_counts())

KeyError: 'audio_path'

In [38]:
print(merged.columns.tolist())


['passage', 'question', 'answer', 'answer_word_count', 'audio_path_x', 'transcript', 'similarity', 'human_label', 'prediction', 'audio_path_y']


In [40]:
print(manifest.columns.tolist())

['passage', 'question', 'answer', 'answer_word_count', 'audio_path']


In [44]:
merged["tts_engine"] = merged["audio_path_x"].apply(lambda p: "gtts" if "gtts" in p else "pyttsx3")

print(merged.groupby("tts_engine")["prediction"].value_counts())

tts_engine  prediction
gtts        Correct       21
            Incorrect      3
            Borderline     1
pyttsx3     Correct       20
            Incorrect      4
            Borderline     1
Name: count, dtype: int64


In [46]:
import pandas as pd
from sklearn.metrics import confusion_matrix

df = pd.read_csv("../data/pipeline_results.csv")
df["final_prediction"] = df["prediction"].apply(lambda p: 1 if p in ["Correct", "Borderline"] else 0)

cm = confusion_matrix(df["human_label"], df["final_prediction"])
print(cm)

[[ 6  1]
 [ 1 42]]


In [48]:
cm_df = pd.DataFrame(
    cm,
    index=["Actual: Incorrect", "Actual: Correct"],
    columns=["Predicted: Incorrect", "Predicted: Correct"]
)
print(cm_df)

                   Predicted: Incorrect  Predicted: Correct
Actual: Incorrect                     6                   1
Actual: Correct                       1                  42


In [50]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [
        accuracy_score(df["human_label"], df["final_prediction"]),
        precision_score(df["human_label"], df["final_prediction"]),
        recall_score(df["human_label"], df["final_prediction"]),
        f1_score(df["human_label"], df["final_prediction"]),
    ]
})
print(metrics_df)

      Metric     Score
0   Accuracy  0.960000
1  Precision  0.976744
2     Recall  0.976744
3   F1 Score  0.976744


In [52]:
examples = pd.concat([
    df[df["prediction"] == "Correct"].sample(3, random_state=1),
    df[df["prediction"] == "Incorrect"].sample(3, random_state=1),
    df[df["prediction"] == "Borderline"],
])
print(examples[["answer", "transcript", "similarity", "prediction"]])

                                               answer  \
5   gelatinous projections edged with cilia that p...   
3   I feel I did the right thing by violating this...   
28                                          the Queen   
40                                           Nintendo   
10                                            Beyoncé   
4                                           10,000 m2   
13                                      Pleurobrachia   
38                                  William Stranahan   

                                           transcript  similarity  prediction  
5   gelatinous projections edged with sillier that...    0.838639     Correct  
3   I feel I did the right thing by violating this...    0.985476     Correct  
28                                         The Queen.    0.969238     Correct  
40                                                念獵獵    0.131477   Incorrect  
10                                         Be on say.    0.167976   Incorrect  
4     

In [54]:
metrics_df.to_csv("../evaluation/metrics_table.csv", index=False)
cm_df.to_csv("../evaluation/confusion_matrix.csv")
examples.to_csv("../evaluation/example_predictions.csv", index=False)

print("همه‌ی فایل‌ها ذخیره شدن")

همه‌ی فایل‌ها ذخیره شدن


In [58]:
import os
print(os.listdir("../evaluation"))

['.gitkeep', 'confusion_matrix.csv', 'evaluate.py', 'example_predictions.csv', 'metrics_table.csv']


In [60]:
from sentence_transformers import SentenceTransformer, util

finetuned_model = SentenceTransformer("models/finetuned_sbert")

# تست روی همون مثال‌های آشنا
sim1 = util.cos_sim(
    finetuned_model.encode("It was founded in 1852"),
    finetuned_model.encode("The museum opened in 1852")
).item()

print(f"Fine-tuned model similarity: {sim1:.3f}")

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6a9ca5c3-0dcc31010f7014236d61095a;33abb269-9897-47e4-849b-609b78c619c2)

Repository Not Found for url: https://huggingface.co/models/finetuned_sbert/resolve/main/modules.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [62]:
from sentence_transformers import SentenceTransformer, util

finetuned_model = SentenceTransformer("../models/finetuned_sbert")

sim1 = util.cos_sim(
    finetuned_model.encode("It was founded in 1852"),
    finetuned_model.encode("The museum opened in 1852")
).item()

print(f"Fine-tuned model similarity: {sim1:.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Fine-tuned model similarity: 0.627


In [64]:
test_pairs = [
    ("Paris", "Paris."),
    ("Lower Lorraine", "Lower the rain."),
    ("gas turbines", "Gast her buns."),
]

for a, b in test_pairs:
    old_sim = util.cos_sim(model.encode(a), model.encode(b)).item()  # مدل قدیمی
    new_sim = util.cos_sim(finetuned_model.encode(a), finetuned_model.encode(b)).item()  # مدل جدید
    print(f"'{a}' vs '{b}': old={old_sim:.3f}, new={new_sim:.3f}")

NameError: name 'model' is not defined

In [66]:
model = SentenceTransformer("all-MiniLM-L6-v2")

test_pairs = [
    ("Paris", "Paris."),
    ("Lower Lorraine", "Lower the rain."),
    ("gas turbines", "Gast her buns."),
]

for a, b in test_pairs:
    old_sim = util.cos_sim(model.encode(a), model.encode(b)).item()
    new_sim = util.cos_sim(finetuned_model.encode(a), finetuned_model.encode(b)).item()
    print(f"'{a}' vs '{b}': old={old_sim:.3f}, new={new_sim:.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'Paris' vs 'Paris.': old=0.963, new=0.968
'Lower Lorraine' vs 'Lower the rain.': old=0.279, new=0.261
'gas turbines' vs 'Gast her buns.': old=0.160, new=0.144
